# Point-in-time workflow

A rolling signal is credible only if its information set is credible. This notebook uses a changing asset universe, an explicit observation calendar, preflight diagnostics, and formation-specific audits. It also demonstrates the difference between an ordinary insufficient window and a hard data-contract failure.

The calendar and membership ledger are synthetic but treated exactly as external point-in-time inputs would be.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mfdro import (
    DataContractError,
    MultiFrequencySignal,
    PathProgress,
    SignalConfig,
    SkipReason,
)

plt.style.use("seaborn-v0_8-whitegrid")
READY_COLOR = "#008c82"
SKIP_COLOR = "#bd5d38"
INK = "#183b4e"

## 1. A globally sparse panel with locally complete universes

Four core assets exist throughout the sample. One asset exits after June 2023; another becomes observable in January 2022 and enters the eligible universe only after a complete 24-month history is available. Global missing values are acceptable because each selected rolling matrix remains full.

In [ ]:
rng = np.random.default_rng(90210)
calendar = pd.bdate_range("2018-01-02", "2024-12-31")
factor = rng.normal(0.00015, 0.0075, size=(len(calendar), 1))
returns = pd.DataFrame(
    factor + rng.normal(0.0, 0.005, size=(len(calendar), 6)),
    index=calendar,
    columns=["core_A", "core_B", "core_C", "core_D", "retired_E", "new_F"],
)
returns.loc[returns.index > pd.Timestamp("2023-06-30"), "retired_E"] = np.nan
returns.loc[returns.index < pd.Timestamp("2022-01-03"), "new_F"] = np.nan

coverage = returns.notna().mean().rename("coverage")
coverage

In [ ]:
availability = returns.notna().astype(int)
figure, axis = plt.subplots(figsize=(10, 3.2))
axis.imshow(availability.T, aspect="auto", interpolation="nearest", cmap="GnBu")
axis.set_yticks(range(len(availability.columns)), labels=availability.columns)
year_starts = [index for index, date in enumerate(calendar) if date.month == 1 and date.day <= 3]
axis.set_xticks(year_starts, labels=[calendar[index].year for index in year_starts])
axis.set(title="Observed return cells", xlabel="year", ylabel="asset")
figure.tight_layout()

## 2. Build the formation-month membership ledger

Membership order is deliberate and becomes part of the numerical audit. In production, this ledger must come from historically recorded eligibility information; MFDRO validates its use, not its economic provenance.

In [ ]:
formation_dates = (
    pd.Series(calendar, index=calendar).groupby(calendar.to_period("M")).max().tolist()
)
core_assets = ["core_A", "core_B", "core_C", "core_D"]
memberships = {}
for formation_date in formation_dates:
    selected = list(core_assets)
    if formation_date <= pd.Timestamp("2023-06-30"):
        selected.append("retired_E")
    if formation_date >= pd.Timestamp("2023-12-29"):
        selected.append("new_F")
    memberships[str(formation_date.to_period("M"))] = selected

pd.Series(
    {month: len(assets) for month, assets in memberships.items()}, name="eligible_assets"
).tail(24)

## 3. Preflight the entire schedule

Preflight constructs every selected window and every frequency measure but skips transport geometry. The first 23 formations cannot contain 24 complete calendar months and are correctly classified as warm-up.

In [ ]:
config = SignalConfig.projected(
    n_projections=64,
    n_quantiles=64,
    random_state=20250301,
)
engine = MultiFrequencySignal(config)
diagnostics = engine.validate_path_inputs(
    returns,
    lookback_months=24,
    memberships=memberships,
    reference_calendar=calendar,
    seed_namespace="point_in_time_demo",
)

assert diagnostics.n_ready + diagnostics.n_insufficient == diagnostics.n_formations
diagnostics.summary()

In [ ]:
formations = diagnostics.formations.copy()
status_colors = formations["status"].map({"ready": READY_COLOR, "insufficient": SKIP_COLOR})
figure, axes = plt.subplots(
    2, 1, figsize=(10, 4.8), sharex=True, gridspec_kw={"height_ratios": [1, 2]}
)
axes[0].scatter(
    formations["date"], np.zeros(len(formations)), color=status_colors, marker="s", s=28
)
axes[0].set_yticks([])
axes[0].set_title("Preflight status: warm-up in rust, ready formations in teal")
axes[1].step(formations["date"], formations["n_assets"], where="mid", color=INK, linewidth=1.8)
axes[1].set(xlabel="formation date", ylabel="selected assets", ylim=(3.5, 6.5))
figure.tight_layout()

## 4. Estimate and retain progress and audit evidence

The callback receives one immutable update per requested formation. It can feed a logger or progress widget without changing numerical behavior.

In [ ]:
progress: list[PathProgress] = []
path = engine.estimate_path(
    returns,
    lookback_months=24,
    memberships=memberships,
    reference_calendar=calendar,
    on_insufficient="skip",
    seed_namespace="point_in_time_demo",
    progress_callback=progress.append,
)

assert len(progress) == diagnostics.n_formations
assert len(path.estimates) == diagnostics.n_ready
assert path.audit["no_future_observations"].all()
assert path.audit["matrix_is_full"].all()
path.summary()

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(path.sqrt_rho.index, path.sqrt_rho, color=INK, linewidth=1.7)
axes[0].set(title="Point-in-time multi-frequency dispersion", ylabel=r"$\sqrt{\rho}$")
for column, color in zip(
    ["n_daily", "n_weekly", "n_monthly"], [INK, READY_COLOR, "#d17a22"], strict=True
):
    axes[1].plot(
        path.audit["date"], path.audit[column], label=column.removeprefix("n_"), color=color
    )
axes[1].set(xlabel="formation date", ylabel="empirical observations")
axes[1].legend(frameon=False, ncol=3)
figure.tight_layout()

## 5. A missing observation date is insufficient, not silently ignored

Removing one date from the source while retaining it in the authoritative calendar produces a machine-readable calendar mismatch. With `skip`, the formation is reported; with `raise`, the same condition becomes an exception.

In [ ]:
missing_date = calendar[(calendar.year == 2024) & (calendar.month == 4)][8]
damaged_returns = returns.drop(index=missing_date)
final_date = pd.Timestamp(formation_dates[-1])
final_month = str(final_date.to_period("M"))
single_membership = {final_month: memberships[final_month]}
calendar_check = engine.validate_path_inputs(
    damaged_returns,
    lookback_months=24,
    formation_dates=[final_date],
    memberships=single_membership,
    reference_calendar=calendar,
)

assert calendar_check.formations.loc[0, "reason"] == SkipReason.REFERENCE_CALENDAR_MISMATCH.value
calendar_check.formations

In [ ]:
try:
    engine.estimate_path(
        damaged_returns,
        lookback_months=24,
        formation_dates=[final_date],
        memberships=single_membership,
        reference_calendar=calendar,
        on_insufficient="raise",
    )
except DataContractError as error:
    calendar_error = str(error)
else:
    raise AssertionError("The calendar mismatch should have raised DataContractError.")

calendar_error

## 6. A selected missing return is a hard failure

Unlike ordinary warm-up, a missing value inside a selected asset-by-date matrix indicates an invalid numerical input. MFDRO never imputes or skips it silently.

In [ ]:
invalid_returns = returns.copy()
invalid_returns.loc[pd.Timestamp("2024-06-17"), "core_A"] = np.nan
try:
    engine.estimate_path(
        invalid_returns,
        lookback_months=24,
        formation_dates=[final_date],
        memberships=single_membership,
        reference_calendar=calendar,
    )
except DataContractError as error:
    missing_value_error = str(error)
else:
    raise AssertionError("A selected missing return should have failed.")

missing_value_error

## 7. Future observations cannot alter a past formation

The final assertion changes every core return after a chosen formation date. The earlier estimate must remain bit-for-bit identical because its selected window ends at formation.

In [ ]:
past_date = pd.Timestamp("2023-06-30")
past_month = str(past_date.to_period("M"))
past_membership = {past_month: memberships[past_month]}
original = engine.estimate_path(
    returns,
    lookback_months=24,
    formation_dates=[past_date],
    memberships=past_membership,
    reference_calendar=calendar,
    on_insufficient="raise",
)
altered_returns = returns.copy()
altered_returns.loc[altered_returns.index > past_date, core_assets] = 0.25
altered = engine.estimate_path(
    altered_returns,
    lookback_months=24,
    formation_dates=[past_date],
    memberships=past_membership,
    reference_calendar=calendar,
    on_insufficient="raise",
)

assert original.estimates.loc[0, "rho"] == altered.estimates.loc[0, "rho"]
original.estimates[["date", "rho", "seed", "n_assets"]]

## Interpretation

The audit establishes what MFDRO can observe: selected dates, selected assets, matrix fullness, sample sizes, seeds, and absence of future rows. It cannot establish that the calendar or membership ledger was genuinely known in the past. Those inputs require independent source governance and must be archived beside the saved signal path.